# Manual Semantic Chunking with Python

This notebook embeds sentences and compares adjacent sentence vectors with cosine similarity. Sentences above the chosen threshold are grouped into the same chunk; lower similarity starts a new chunk. This is useful when fixed-size splitting breaks related ideas apart, and makes the chunk-boundary logic and threshold visible for experimentation.

In [44]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [45]:
text="""
LangChain is a framework used to build applications powered by Large Language Models (LLMs). It provides document loaders, text splitters, retrievers, and prompt templates for creating RAG applications.
LangChain supports integration with different LLM providers and vector databases. Developers can use it to retrieve relevant documents and provide contextual information to language models.
Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data analysis, and artificial intelligence.
Python supports object-oriented, procedural, and functional programming paradigms. Popular libraries such as Pandas, NumPy, and Scikit-learn help developers process data and build machine learning models.
The Taj Mahal is a famous historical monument located in Agra, India. Built by Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal, it is renowned for its white marble architecture.
Photosynthesis is the process by which green plants use sunlight, water, and carbon dioxide to produce glucose and release oxygen. This process is essential for sustaining life and maintaining oxygen levels in Earth's atmosphere.
"""

In [46]:
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6955.84it/s]


In [47]:
#Step1 - 
sentences=[ s.strip() for s in text.split("\n") if s.strip()]

#STep2-

embedding = model.encode(sentences)

#Step3 - Threshold
threshold = 0.7
chunks=[]
current_chunk=[sentences[0]]

embedding


array([[-0.04633592, -0.01460671,  0.02500644, ...,  0.00518775,
         0.12363064,  0.03393371],
       [-0.00469164, -0.04657001,  0.01926512, ...,  0.01373802,
         0.0938801 ,  0.00981863],
       [-0.04130879,  0.01262208, -0.01830262, ...,  0.17788793,
         0.14375089,  0.00286688],
       [-0.03756259, -0.02911105, -0.02440262, ...,  0.07660335,
         0.1318975 ,  0.03424191],
       [ 0.01547157,  0.09565173, -0.02450867, ...,  0.04366896,
        -0.0230135 , -0.05583533],
       [-0.06837911,  0.09083023, -0.07269458, ...,  0.05283272,
         0.11958952,  0.00411925]], shape=(6, 384), dtype=float32)

In [48]:
#Step4 - Merge cosin similar chunk
threshold = 0.5
chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):
    sim = cosine_similarity([embedding[i - 1]], [embedding[i]])[0][0]

    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

chunks.append(" ".join(current_chunk))

In [49]:
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}\n: {chunk}")


Chunk 1
: LangChain is a framework used to build applications powered by Large Language Models (LLMs). It provides document loaders, text splitters, retrievers, and prompt templates for creating RAG applications. LangChain supports integration with different LLM providers and vector databases. Developers can use it to retrieve relevant documents and provide contextual information to language models.

Chunk 2
: Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data analysis, and artificial intelligence. Python supports object-oriented, procedural, and functional programming paradigms. Popular libraries such as Pandas, NumPy, and Scikit-learn help developers process data and build machine learning models.

Chunk 3
: The Taj Mahal is a famous historical monument located in Agra, India. Built by Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal, it is renowned for its white marble arc